# Setting Up MR-LFADS

This tutorial walks through how to configure the MR-LFADS inference pipeline.

As a concrete example, we train MR-LFADS on the synthetic *Memory* network described in our paper.

## 1. Download Example Dataset

To follow along, download the dataset from Zenodo:

https://zenodo.org/records/19444085

After downloading, place the `.h5` file at: `<datapath>/memory_network_icml2025/data.h5`. 

Here, `<datapath>` refers to the directory defined in `config/paths.py`.

## 2. Configuration Structure

The corresponding configuration for this example is located in `examples/01_memory_network`.

MR-LFADS uses Hydra to organize configuration files hierarchically:

```text
main.yaml
├── model/model.yaml
├── datamodule/datamodule.yaml
└── callbacks/callbacks.yaml
```

Each component defines a different part of the training pipeline:
- `model/`: model architecture and parameters  
- `datamodule/`: data loading and preprocessing  
- `callbacks/`: training-time utilities (e.g., checkpointing, logging)

The entry point is `main.yaml`, which specifies the model, datamodule, and callbacks configurations, along with additional settings such as trainer parameters, TensorBoard logging, and random seed.

## 3. Model Configuration (model/model.yaml)

The model configuration consists of three main categories of parameters:

### (1). Data Specification

These parameters define the structure of the input data, for example:

* `num_other_areas`: total number of areas − 1
* `seq_len`: number of time steps per trial
* `ic_enc_seq_len`: number of time steps used to infer the initial condition (IC)

### (2). Model Architecture

These parameters define the structure of the SRLFADS model:

* `areas_params`: specifies architecture details such as 
    - generator dimensionality
    - inter-area communication dimensions
    - output distribution

For the full list of available parameters, refer to `mrlfads.model.SRLFADS`.

### (3). Regularization

These parameters control regularization, which are key hyperparameters for MR-LFADS:

* L2 regularization (`l2_*`): weight decay applied to GRU parameters
* KL regularization (`kl_*`): regularization on inferred inputs and inter-area messages

For the full list of available parameters, refer to `mrlfads.model.MRLFADS`.

## 4. Datamodule Configuration (datamodule/datamodule.yaml)

The following parameters are required for configuring the datamodule:

* `filename`: path to the `data.h5` file (relative to `config.paths.datapath`)
* `area_names`: list of brain area names corresponding to the datasets (e.g., ["M1", "PMd"])
* `session_idxs`: list of session indices to include (e.g., [0] for the first session)
* `time_dim`: number of time steps per trial

For additional parameters and options, see `mrlfads.datamodules.BasicDataModule`.

## 5. Callbacks Configuration (callbacks/callbacks.yaml)

Callbacks are managed via the `mrlfads.callbacks.OnEpochEndCalls` class, which executes specified callback functions at the end of each training or validation epoch. This enables logging, visualization, and intermediate analysis during training, and can be easily extended with custom callbacks. The following callbacks are provided in `mrlfads/callbacks.py`:

* `Log`: Logs training statistics such as learning rate
* `InferredRatesPlot`: Plots inferred firing rates alongside ground-truth spike data (optionally smoothed for comparison)
* `InferredPredsPlot`: Similar to `InferredRatesPlot`, but for held-out neurons
* `ProctorSummaryPlot`: Visualizes training and validation metrics (e.g., loss, KL divergence)
* `AnatomyPlot`: Displays the inferred inter-area communication structure

## 6. Test Run

To train MR-LFADS, run:

In [ ]:
import os
import config.paths as path

# Path to the folder that contains main.yaml, relative to config.paths.homepath
workdir = os.path.join(path.homepath, "examples", "01_memory_network")

In [ ]:
import subprocess

subprocess.run(["python", "exec.py"], cwd=workdir, check=True)